# 7 - Number of laps (nLaps)

In [1]:
%%capture
import os

# go to the project root (the notebook is in the "notebooks" folder)
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

# run the graphics part (which runs the reading part) to get x_c, y_c, r and theta
# %%capture hides its figures and prints (to remove when we merge the notebooks)
# get_ipython().run_line_magic("run", ...) is the same as %run, but works inside %%capture
if "theta_trials" not in globals():
    get_ipython().run_line_magic("run", "notebooks/02_graphics.ipynb")      

### Keep each record from the first entry into the target
As in MouseReMoCo, the statistics are computed only after the cursor enters
the target for the first time (first sample with mouseInTarget = 1).
We reuse the centered coordinates, r and θ computed in notebook 2 (02_graphics).

In [2]:
# data of each record, from the first entry into the target
t_kept, x_kept, y_kept, r_kept, theta_kept = [], [], [], [], []

for i in range(n_records):
    first = np.argmax(in_target_trials[i] == 1)   # index of the first sample inside the target

    t_kept.append(t_trials[i][first:])
    x_kept.append(x_c_trials[i][first:])       # centered x (from notebook 2)
    y_kept.append(y_c_trials[i][first:])       # centered y, pointing up (from notebook 2)
    r_kept.append(r_trials[i][first:])         # distance to the center (from notebook 2)
    theta_kept.append(theta_trials[i][first:]) # angle (from notebook 2)

    print(f"record {i}: first entry at {t_trials[i][first]:7.3f} s (sample {first}), "
          f"{len(t_kept[i])} samples kept")

record 0: first entry at   0.022 s (sample 0), 3971 samples kept
record 1: first entry at  40.966 s (sample 92), 3806 samples kept
record 2: first entry at  80.832 s (sample 58), 3840 samples kept
record 3: first entry at 120.908 s (sample 52), 3832 samples kept
record 4: first entry at 160.908 s (sample 93), 3843 samples kept


### Observations
- Record 0 starts directly inside the target (sample 0), so all its samples are kept.
- In records 1 to 4, the cursor enters the target after about 1 s (92, 58, 52 and 93 samples removed).
- About 3800 to 3970 samples are kept per record, so almost the whole 20 s of each record is used for the statistics.

### Number of laps
θ (computed in notebook 2 with arctan2) is the angle of the cursor around the center, between -π and π.
When the cursor crosses the left side of the circle, θ jumps from +π to -π
(e.g. 170°, 180°, -170° instead of 170°, 180°, 190°).
np.unwrap removes these jumps, so that θ keeps increasing (or decreasing) continuously.
The number of laps is then the total change of angle divided by one full turn (2π):
nLaps = (θ_end - θ_start) / 2π. A negative value means that θ decreases.

In [3]:
n_laps = []

for i in range(n_records):
    theta = np.unwrap(theta_kept[i])   # continuous angle (radians)
    n_laps.append((theta[-1] - theta[0]) / (2 * np.pi))  # total turn / one full turn

    print(f"record {i}: nLaps = {n_laps[i]:6.2f}")

record 0: nLaps = -10.84
record 1: nLaps = -11.97
record 2: nLaps = -13.26
record 3: nLaps = -12.83
record 4: nLaps = -14.74


### Results
We find the same values as in the "recomputed statistics" of the assignment (-10.84, -11.97, -13.26, -12.83, -14.74). They differ from the MouseReMoCo table by at most 0.01 lap (e.g. -10.84 vs -10.85), due to rounding.
The values are negative because we use a y axis pointing up (as in notebook 2):
θ decreases, so the cursor turns clockwise on the screen.
With a y axis pointing down (screen coordinates), the same movement would give positive values,
as in the former version of the assignment.

The number of laps increases overall over the session (from about 11 to 15 laps in 20 s),
but not at every record: record 3 (-12.83) has fewer laps than record 2 (-13.26).

# Compute Re, Te and error

Re stands for Radius effective which means the radius of the trajectory that the participant really follow. 

Te stands for Tolerance effective which means the width of the cursor during the task. 

Error is the difference between the trajectory that the participant should follow and the trajectpry that he really follows. 

### Compute Re

To compute Re, we simply compute the mean radius values for each record. This method is used in the textbook: https://perso.lisn.upsaclay.fr/chapuis/ihm-polytech/pointage.pdf, which explains Fitts' law. 


In [6]:
Re = []

for i in range(n_records):
    first = np.argmax(in_target_trials[i] == 1)  # first sample inside the target (as in task 7)
    r = r_trials[i][first:]                      # distance to the center, beginning removed (notebook 2)
    in_target_kept = in_target_trials[i][first:] # mouseInTarget, beginning removed
    
    Re.append(np.mean(r))  # mean distance to the center, beginning removed
    print(f"record {i}: Re = {Re[i]:6.2f} px")

record 0: Re = 213.86 px
record 1: Re = 209.85 px
record 2: Re = 209.43 px
record 3: Re = 209.46 px
record 4: Re = 210.89 px


By using this method, we obtain exactly the same results as in the instructions.